# 02U — Model Development (Updated)
**Fixes:** Label Leakage | Cross-Domain Negative Pairs | Vocabulary OOV
**Skip:** CV synthetic length improvement

In [60]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from src.preprocessing.nlp_preprocessor import NLPPreprocessor
from src.preprocessing.embeddings import EmbeddingManager
from src.models.model_architecture import SkillAlignMatcher
from src.models.custom_loss import focal_loss
from src.training.train import ModelTrainer
from src.utils.metrics import compute_all_metrics, compute_classification_report, check_performance_targets
from src.utils.visualization import TrainingVisualizer

print(f'TF: {tf.__version__}')
print('Imports OK!')

TF: 2.21.0
Imports OK!


## 1. Load Raw Data

In [61]:
# Load job postings
df_posting = pd.read_csv(
    '../Dataset/database_design/job_posting.csv',
    usecols=['job_posting_id', 'title', 'job_description']
).dropna(subset=['job_description'])
print(f'Job postings: {len(df_posting):,}')

# Load job_skills
df_job_skills = pd.read_csv('../Dataset/database_design/job_skills.csv')
df_skill_names = pd.read_csv('../Dataset/database_design/skills.csv')
df_job_skills = df_job_skills.merge(df_skill_names, on='skill_id', how='left')
job_skills_grouped = (
    df_job_skills.groupby('job_posting_id')['skill_name']
    .apply(list).reset_index()
)
job_skills_grouped.columns = ['job_posting_id', 'required_skills']

# Load industries untuk cross-domain grouping
df_job_ind = pd.read_csv('../Dataset/database_design/job_industries.csv')
df_industries = pd.read_csv('../Dataset/database_design/industries.csv')
df_job_ind = df_job_ind.merge(df_industries, on='industry_id', how='left')
ind_per_job = df_job_ind.groupby('job_posting_id')['industry_name'].first().reset_index()

# Gabungkan semua
df = df_posting.merge(job_skills_grouped, on='job_posting_id', how='inner')
df = df.merge(ind_per_job, on='job_posting_id', how='left')
df['industry_name'] = df['industry_name'].fillna('Other')
df = df.dropna(subset=['job_description'])
print(f'Combined shape: {df.shape}')
print(f'Unique industries: {df["industry_name"].nunique()}')

Job postings: 123,842
Combined shape: (122090, 5)
Unique industries: 375


## 2. FIX #1 — Extract Keywords dari Job Description (bukan 35 generic skills)

In [62]:
SAMPLE_SIZE = min(len(df), 30000)
df_sample = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# Fit TF-IDF pada job descriptions asli
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=5
)
tfidf.fit(df_sample['job_description'].str[:2000])
feature_names = np.array(tfidf.get_feature_names_out())
print(f'TF-IDF vocab: {len(feature_names)}')

def extract_top_keywords(text, tfidf_model, feat_names, topn=10):
    try:
        vec = tfidf_model.transform([str(text)[:2000]])
        scores = vec.toarray()[0]
        top_idx = scores.argsort()[::-1][:topn]
        return [feat_names[i] for i in top_idx if scores[i] > 0]
    except:
        return []

# Test
sample_kw = extract_top_keywords(df_sample['job_description'].iloc[0], tfidf, feature_names, 6)
print(f'Sample keywords: {sample_kw}')

TF-IDF vocab: 5000
Sample keywords: ['leasing', 'resident', 'marketing', 'property', 'occupancy', 'property management']


## 3. FIX #2 — Cross-Domain Negative Pairing

**Sebelumnya:** cv_neg ↔ job_desc (SAMA) → label leakage

**Fix:** cv_pos dari job A (domain X) dipasangkan dengan job B dari **domain berbeda Y**

In [63]:
industry_groups = df_sample.groupby('industry_name')
industry_to_indices = {
    ind: grp.index.tolist()
    for ind, grp in industry_groups
    if len(grp) >= 3
}
valid_industries = list(industry_to_indices.keys())
print(f'Industries with >= 3 jobs: {len(valid_industries)}')

np.random.seed(42)
cv_texts, job_texts, labels = [], [], []
skipped = 0

for idx, row in df_sample.iterrows():
    job_desc   = str(row['job_description'])[:2000]
    req_skills = row['required_skills'] if isinstance(row['required_skills'], list) else []
    title      = str(row.get('title', 'Professional'))
    industry   = row['industry_name']

    # FIX #1: Extract keywords dari job_description asli
    job_keywords  = extract_top_keywords(job_desc, tfidf, feature_names, topn=8)
    all_job_skills = list(set(req_skills + job_keywords))

    if len(all_job_skills) < 2:
        skipped += 1
        continue

    # === POSITIVE PAIR ===
    n_pick = max(2, int(len(all_job_skills) * np.random.uniform(0.6, 1.0)))
    picked = np.random.choice(all_job_skills, size=min(n_pick, len(all_job_skills)), replace=False).tolist()
    cv_pos = (
        f"Experienced {title} with skills in {', '.join(picked[:5])}. "
        f"Strong background in {picked[0]}. "
        f"{np.random.randint(2, 10)} years of professional experience."
    )
    cv_texts.append(cv_pos)
    job_texts.append(job_desc)
    labels.append(1)

    # === NEGATIVE PAIR — FIX #2: Cross-Domain ===
    other_inds = [i for i in valid_industries if i != industry]
    if not other_inds:
        other_inds = valid_industries
    neg_industry = np.random.choice(other_inds)
    neg_idx      = np.random.choice(industry_to_indices[neg_industry])
    neg_job_desc = str(df_sample.loc[neg_idx, 'job_description'])[:2000]

    cv_texts.append(cv_pos)       # CV sama
    job_texts.append(neg_job_desc)  # Job dari domain berbeda
    labels.append(0)

labels = np.array(labels, dtype=np.float32)
print(f'Pairs: {len(labels):,} | Positive: {int(labels.sum()):,} | Negative: {int(len(labels)-labels.sum()):,}')
print(f'Balance: {labels.mean():.2%} | Skipped: {skipped}')

Industries with >= 3 jobs: 221
Pairs: 59,998 | Positive: 29,999 | Negative: 29,999
Balance: 50.00% | Skipped: 1


## 4. FIX #3 — Vocabulary dari Job Descriptions Asli (bukan hanya CV synthetic)

In [64]:
MAX_VOCAB_SIZE = 15000  # Naik dari 10000
MAX_SEQ_LEN   = 300
EMBEDDING_DIM = 128

preprocessor = NLPPreprocessor(
    max_vocab_size=MAX_VOCAB_SIZE,
    max_seq_len=MAX_SEQ_LEN,
    use_lemmatizer=True,
    language='english'
)

# FIX #3: Fit pada CV + Job descriptions asli untuk pastikan kata teknikal masuk vocab
extra_job_texts = df_sample['job_description'].str[:2000].tolist()
all_fit_texts   = cv_texts + job_texts + extra_job_texts
preprocessor.fit(all_fit_texts)
print(f'Vocabulary size: {preprocessor.vocab_size}')

# Cek kata teknikal
word_index = preprocessor.word_index
tech_check = ['python', 'react', 'docker', 'tensorflow', 'javascript', 'kubernetes']
print('Tech terms in vocab:')
for w in tech_check:
    idx = word_index.get(w)
    print(f'  {w}: {"idx=" + str(idx) if idx else "NOT FOUND"}')

Vocabulary size: 15000
Tech terms in vocab:
  python: idx=1171
  react: idx=2335
  docker: idx=3669
  tensorflow: idx=8820
  javascript: idx=1940
  kubernetes: idx=3004


In [65]:
cv_sequences  = preprocessor.transform(cv_texts)
job_sequences = preprocessor.transform(job_texts)
print(f'CV: {cv_sequences.shape} | Job: {job_sequences.shape}')

CV: (59998, 300) | Job: (59998, 300)


## 5. Word2Vec Embedding

In [66]:
tokenized = [t.split() for t in preprocessor.preprocess_batch(all_fit_texts)]
emb_manager = EmbeddingManager(embedding_dim=EMBEDDING_DIM, min_count=2, window=5)
emb_manager.train_word2vec(tokenized, epochs=20, sg=1)

embedding_matrix = emb_manager.create_embedding_matrix(
    word_index=preprocessor.word_index,
    vocab_size=preprocessor.vocab_size
)
print(f'Embedding matrix: {embedding_matrix.shape}')

for w in ['python', 'react', 'docker']:
    sim = emb_manager.get_similar_words(w, topn=3)
    if sim:
        print(f'  "{w}": {[(x[0], round(x[1],2)) for x in sim]}')

Embedding matrix: (15000, 128)
  "python": [('java', 0.76), ('javascript', 0.75), ('sql', 0.74)]
  "react": [('j', 0.78), ('typescript', 0.73), ('javascript', 0.73)]
  "docker": [('kubernetes', 0.86), ('runtimes', 0.74), ('aws', 0.72)]


## 6. Train-Val-Test Split

In [67]:
cv_train, cv_test, job_train, job_test, y_train, y_test = train_test_split(
    cv_sequences, job_sequences, labels,
    test_size=0.2, random_state=42, stratify=labels
)
cv_train, cv_val, job_train, job_val, y_train, y_val = train_test_split(
    cv_train, job_train, y_train,
    test_size=0.15, random_state=42, stratify=y_train
)
print(f'Train: {len(y_train):,} | Val: {len(y_val):,} | Test: {len(y_test):,}')

Train: 40,798 | Val: 7,200 | Test: 12,000


## 7. Build & Train Model

In [68]:
matcher = SkillAlignMatcher(
    vocab_size=preprocessor.vocab_size,
    max_seq_len=MAX_SEQ_LEN,
    embedding_dim=EMBEDDING_DIM,
    attention_units=128,
    embedding_matrix=embedding_matrix,
    trainable_embedding=True
)
model = matcher.build_model()
model.summary()

Model: "SkillAlign_Matcher"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cv_input            │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_input           │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_embedding    │ (None, 300, 128)  │  1,920,000 │ cv_input[0][0],   │
│ (Embedding)         │                   │            │ job_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_conv1d_1         │ (None, 300, 128)  │     49,280 │ shared_embedding… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_conv1d_1        │ (None, 300, 128)  │     49,280 │ shared_embedding… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_bn_1             │ (None, 300, 128)  │        512 │ cv_conv1d_1[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_bn_1            │ (None, 300, 128)  │        512 │ job_conv1d_1[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_conv1d_2         │ (None, 300, 64)   │     24,640 │ cv_bn_1[0][0]     │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_conv1d_2        │ (None, 300, 64)   │     24,640 │ job_bn_1[0][0]    │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_bn_2             │ (None, 300, 64)   │        256 │ cv_conv1d_2[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_bn_2            │ (None, 300, 64)   │        256 │ job_conv1d_2[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ custom_attention    │ (None, 128)       │     24,960 │ cv_bn_2[0][0],    │
│ (CustomAttentionLa… │                   │            │ job_bn_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cv_global_pool      │ (None, 64)        │          0 │ cv_bn_2[0][0]     │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ job_global_pool     │ (None, 64)        │          0 │ job_bn_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_merge       │ (None, 256)       │          0 │ custom_attention… │
│ (Concatenate)       │                   │            │ cv_global_pool[0… │
│                     │                   │            │ job_global_pool[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     65,792 │ feature_merge[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_bn_1          │ (None, 256)       │      1,024 │ dense_1[0][0]   

 Total params: 2,202,881 (8.40 MB)

 Trainable params: 2,201,345 (8.40 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [69]:
trainer = ModelTrainer(
    model=model,
    config={
        'batch_size': 64,
        'epochs': 50,
        'learning_rate': 0.001,
        'early_stopping_patience': 8,
        'reduce_lr_patience': 4,
        'f1_patience': 7,
        'f1_threshold': 0.75,
        'focal_loss_gamma': 2.0,
        'focal_loss_alpha': 0.25
    },
    log_dir='../logs/training_v2',
    model_dir='../models'
)
trainer.compile_model()
print('Compiled. Training...')

history = trainer.train(
    x_train=[cv_train, job_train],
    y_train=y_train,
    x_val=[cv_val, job_val],
    y_val=y_val
)
print('Training complete!')

Compiled. Training...
Epoch 1/50
638/638 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.5473 - auc: 0.6025 - loss: 0.1062 - precision: 0.6076 - recall: 0.2755
Epoch 1: val_accuracy improved from None to 0.74028, saving model to ../models\best_model.keras

  [F1Callback] Epoch 1: F1=0.6979 | Precision=0.8340 | Recall=0.6000 | Best F1=0.0000
  [F1Callback] Model saved to ../models\best_f1_model.keras (F1=0.6979)
638/638 ━━━━━━━━━━━━━━━━━━━━ 77s 116ms/step - accuracy: 0.5909 - auc: 0.7014 - loss: 0.0789 - precision: 0.6997 - recall: 0.3185 - val_accuracy: 0.7403 - val_auc: 0.8756 - val_loss: 0.0475 - val_precision: 0.8340 - val_recall: 0.6000 - learning_rate: 0.0010 - val_f1_score: 0.6979 - val_precision_custom: 0.8340 - val_recall_custom: 0.6000
Epoch 2/50
638/638 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.7364 - auc: 0.8826 - loss: 0.0467 - precision: 0.8580 - recall: 0.5672
Epoch 2: val_accuracy improved from 0.74028 to 0.85556, saving model to ../models\best_model.keras

  [F1

## 8. Evaluation

In [70]:
viz = TrainingVisualizer(save_dir='../logs/plots_v2')
viz.plot_training_history(history, filename='training_history_v2.png')

y_pred = model.predict([cv_test, job_test], verbose=0)
metrics = compute_all_metrics(y_test, y_pred)

print('=== Metrics ===')
for k, v in metrics.items():
    print(f'  {k}: {v}')

print('\n=== Classification Report ===')
print(compute_classification_report(y_test, y_pred))

print('\n=== Performance Targets ===')
targets = check_performance_targets(metrics)
for name, info in targets.items():
    status = 'PASS' if info['passed'] else 'FAIL'
    print(f"  {name}: {info['value']} (target: {info['target']}) [{status}]")

=== Metrics ===
  accuracy: 0.9003
  precision: 0.9266
  recall: 0.8695
  f1_score: 0.8972
  mae: 0.2427
  auc: 0.9678

=== Classification Report ===
              precision    recall  f1-score   support

   Not Match       0.88      0.93      0.90      6000
       Match       0.93      0.87      0.90      6000

    accuracy                           0.90     12000
   macro avg       0.90      0.90      0.90     12000
weighted avg       0.90      0.90      0.90     12000


=== Performance Targets ===
  accuracy: 0.9003 (target: >= 0.85) [PASS]
  mae: 0.2427 (target: <= 0.02) [FAIL]


In [71]:
y_bin = (y_pred > 0.5).astype(int).flatten()
viz.plot_confusion_matrix(y_test, y_bin, filename='cm_v2.png')
viz.plot_score_distribution(y_pred.flatten(), y_test, filename='score_dist_v2.png')
print('Plots saved.')

Plots saved.


## 9. Sanity Check — Verifikasi High vs Low Match

In [72]:
quick_tests = [
    {
        'label': 'HIGH MATCH (Data Scientist)',
        'cv':  'Experienced Data Scientist with 5 years in Python TensorFlow machine learning deep learning and data analysis.',
        'job': 'Looking for a Data Scientist with strong Python skills experience in ML frameworks TensorFlow and statistical analysis.'
    },
    {
        'label': 'LOW MATCH (Marketing vs Data Analyst)',
        'cv':  'Marketing Manager with 7 years experience in digital marketing SEO SEM social media management and campaign analytics.',
        'job': 'Data Analyst position requiring SQL Python Tableau statistical analysis and business intelligence.'
    },
    {
        'label': 'VERY LOW (Designer vs Developer)',
        'cv':  'UI/UX Designer with expertise in Figma Adobe XD user research prototyping and design systems. No coding experience.',
        'job': 'Frontend Developer position requiring React TypeScript JavaScript CSS and component-based architecture.'
    }
]

print('=== Sanity Check (skor harus turun dari high ke very low) ===')
for t in quick_tests:
    cv_seq  = preprocessor.process(t['cv'])
    job_seq = preprocessor.process(t['job'])
    score   = model.predict([cv_seq, job_seq], verbose=0)[0][0]
    print(f"  [{t['label']}] Score: {score:.4f}")

=== Sanity Check (skor harus turun dari high ke very low) ===
  [HIGH MATCH (Data Scientist)] Score: 0.0186
  [LOW MATCH (Marketing vs Data Analyst)] Score: 0.3860
  [VERY LOW (Designer vs Developer)] Score: 0.5734


## 10. Save Artifacts

In [73]:
model_path = trainer.save_model('skillalign_matcher_v2.keras')
print(f'Model: {model_path}')

os.makedirs('../preprocessors', exist_ok=True)
joblib.dump(preprocessor, '../preprocessors/nlp_preprocessor_v2.pkl')
print('Preprocessor: ../preprocessors/nlp_preprocessor_v2.pkl')

emb_manager.save_model('../preprocessors/embedding_manager_v2.pkl')
print('Embedding: ../preprocessors/embedding_manager_v2.pkl')

config = matcher.get_model_config()
config['version'] = 'v2'
config['fixes'] = [
    'cross_domain_negative_pairs',
    'tfidf_keyword_extraction',
    'expanded_vocabulary_15k'
]
config['metrics'] = metrics
with open('../models/model_config_v2.json', 'w') as f:
    json.dump(config, f, indent=2, default=str)
print('Config: ../models/model_config_v2.json')
print('\n=== Done! ===')

Model: ../models\skillalign_matcher_v2.keras
Preprocessor: ../preprocessors/nlp_preprocessor_v2.pkl
Embedding: ../preprocessors/embedding_manager_v2.pkl
Config: ../models/model_config_v2.json

=== Done! ===
